# EPCLM-008 T4 Resource & Recovery Validation
Select Tesla T4, enable Internet and grant GITHUB_TOKEN. Save & Run All runs fresh engineering fixtures and publishes the reports. The checkpoint archive is retained in Kaggle Output.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import tempfile
import uuid
import zipfile
import hashlib

remote = 'https://github.com/ArcheLabs/mini-cells.git'
branch = 'research/epclm008-full-attention-system-benchmark'
with tempfile.TemporaryDirectory(prefix='epclm008-') as directory:
    repo = Path(directory) / 'repo'
    subprocess.run(['git', 'clone', '--depth=1', '--branch', branch, remote, str(repo)],
                   check=True, timeout=300)
    source = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip()
    sys.path.insert(0, str(repo))
    from scripts.research.publish_experiment_results import (
        ExperimentSpec, load_github_token, prepare_artifacts, push_results,
    )
    load_github_token('GITHUB_TOKEN')
    import torch
    if not torch.cuda.is_available() or 'T4' not in torch.cuda.get_device_name():
        raise RuntimeError('Select a Tesla T4 accelerator.')
    base = repo / 'research/projects/cellular-language-models/experiments/epclm-008-full-attention-system-benchmark'
    requirements = base.parent / 'epclm-004-unified-memory-10m/m4/requirements.txt'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)],
                   check=True, timeout=300)
    run_id = f'epclm008-t4-resource-{source[:12]}-{uuid.uuid4().hex[:8]}'
    output = Path('/kaggle/working') / run_id
    env = {**os.environ, 'OMP_NUM_THREADS': '1', 'CUBLAS_WORKSPACE_CONFIG': ':4096:8'}
    output.mkdir()
    log = output / 'raw.log'
    error = None
    try:
        with log.open('w') as stream:
            subprocess.run([sys.executable, str(base / 'resource008.py'), str(output / 'measurements')],
                           cwd=repo, env=env, stdout=stream, stderr=subprocess.STDOUT,
                           check=True, timeout=6600)
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as exc:
        error = str(exc)
    for report in (output / 'measurements').glob('*.json'):
        shutil.copy2(report, output / report.name)
    decision_path = output / 'decision.json'
    resource = output / 'resource.json'
    results = json.loads(resource.read_text()) if resource.exists() else {}
    recovery = results.get('recovery', {})
    measurements = [row['status'] for arm in results.get('measurements', {}).values()
                    for row in arm.values()]
    decision = {'status': 'INCONCLUSIVE', 'test': 'SEALED', 'official_updates': 0,
                'engineering_preflight': 'FAIL' if error or 'FAIL' in measurements
                or len(recovery) != 3 or any(r['status'] != 'PASS' for r in recovery.values())
                else 'PASS', 'error': error,
                'resource_coverage': 'ALL_FIT' if measurements == ['PASS'] * 9 else 'PARTIAL'}
    archive = output / 'engineering-checkpoints.zip'
    with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_STORED) as bundle:
        for arm, record in recovery.items():
            if record['status'] == 'PASS':
                name = record['checkpoint']
                bundle.write(output / 'measurements' / name, name)
    with zipfile.ZipFile(archive) as bundle:
        for record in recovery.values():
            if record['status'] == 'PASS':
                with bundle.open(record['checkpoint']) as stream:
                    assert hashlib.file_digest(stream, 'sha256').hexdigest() == record['sha256']
    with archive.open('rb') as stream:
        archive_sha = hashlib.file_digest(stream, 'sha256').hexdigest()
    archive_report = {
        'path': archive.name, 'bytes': archive.stat().st_size, 'sha256': archive_sha,
        'storage': 'Kaggle saved Output', 'members_verified': True,
        'remote_readback': 'PENDING',
    }
    if any(record['status'] == 'PASS' for record in recovery.values()):
        try:
            import requests
            with requests.Session() as session:
                session.headers.update({'Authorization': 'Bearer ' + load_github_token('GITHUB_TOKEN'),
                                        'Accept': 'application/vnd.github+json'})
                api = 'https://api.github.com/repos/ArcheLabs/mini-cells/releases'
                release = session.post(api, json={'tag_name': run_id, 'target_commitish': source,
                    'name': run_id, 'prerelease': True}, timeout=(15, 60))
                release.raise_for_status()
                release = release.json()
                with archive.open('rb') as stream:
                    asset = session.post(release['upload_url'].split('{', 1)[0],
                        params={'name': archive.name}, data=stream,
                        headers={'Content-Type': 'application/zip'}, timeout=(15, 300))
                asset.raise_for_status()
                asset = asset.json()
                with session.get(api + '/assets/' + str(asset['id']), stream=True,
                                 headers={'Accept': 'application/octet-stream'}, timeout=(15, 300)) as response:
                    response.raise_for_status()
                    digest, size = hashlib.sha256(), 0
                    for chunk in response.iter_content(1024 * 1024):
                        digest.update(chunk)
                        size += len(chunk)
                if digest.hexdigest() != archive_sha or size != archive_report['bytes']:
                    raise ValueError('Remote checkpoint archive hash or size differs')
                archive_report.update(remote_readback='PASS', asset_id=asset['id'],
                                      download_url=asset['browser_download_url'])
        except Exception as exc:
            archive_report.update(remote_readback='FAIL', error=str(exc))
            decision['engineering_preflight'] = 'FAIL'
    (output / 'archive.json').write_text(json.dumps(archive_report, indent=2))
    result_branch = f'kaggle/{run_id}'
    decision.update(format='minicells.epclm008.resource.v1', run_id=run_id,
                    source_commit=source, gpu=torch.cuda.get_device_name(),
                    gpu_total_bytes=torch.cuda.get_device_properties(0).total_memory,
                    branch=result_branch)
    decision_path.write_text(json.dumps(decision, indent=2))
    stage = repo / 'results/epclm008-resource'
    stage.mkdir(parents=True)
    for report in [*output.glob('*.json'), log]:
        shutil.copy2(report, stage / report.name)
    spec = ExperimentSpec('results/epclm008-resource', f'artifacts/experiments/epclm008/{run_id}',
                          tuple(sorted(p.name for p in stage.iterdir())),
                          result_branch, decision['format'])
    try:
        destination = prepare_artifacts(repo, 'epclm008', spec, None)
        push_results(repo, destination, 'epclm008', result_branch, 'GITHUB_TOKEN', 'origin/main')
        sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip()
        receipt = {'status': 'REMOTE_SHA_VERIFIED', 'branch': result_branch, 'commit': sha}
    except Exception:
        (output / 'publication.json').write_text(json.dumps({'status': 'PUBLICATION_FAILED'}))
        raise
    (output / 'publication.json').write_text(json.dumps(receipt, indent=2))
    print(decision, flush=True)
    print(receipt, flush=True)
    if error or decision['engineering_preflight'] != 'PASS':
        raise RuntimeError(f'Preflight failed. Inspect {decision_path}.')
